# Choosing fewer b-values: can we maintain the accuracy of **your** diffusion protocol with fewer measurements?

In the previous session, you designed a diffusion MRI acquisition sequence and generated a **list of b-values**. Each additional measurement increases scan time, which is particularly important for placental MRI.

**The challenge: Can we identify a smaller subset of your b-values that estimates diffusion parameters almost as accurately as the full protocol?**

We'll use [TADRED](https://github.com/PaddySlator/tadred), through ED_MRI, to optimise your protocol. TADRED is a machine-learning method that identifies the most informative measurements for a particular task from a larger set of candidate acquisitions.

Here, TADRED will use simulated signals at **all your b-values** to select a smaller subset and train a neural network to estimate diffusion parameters from those measurements.

We'll treat the TADRED optimiser as a black box. Your job is to **set up the problem, run the optimisation, and critically evaluate whether the selected measurements actually improve performance**.


In [ ]:
# '''ONLY ON GOOGLE COLAB: remove the leading "# " of the five lines below and run this cell once.
# In the MRITogetherThon environment ED_MRI and TADRED are already installed: leave it as it is.
# (The commits are the ones of the TogetherThon image; drop the checkouts to use the latest versions.)'''
# !git clone https://github.com/sbb-gh/ED_MRI && git -C ED_MRI checkout 070c387d673618c66d76f02b42950afbb9acf565
# !git clone https://github.com/PaddySlator/tadred.git && git -C tadred checkout 781ab5eb9301d2a01dd1065b9d9de7d3a8c9a003
# import sys
# !{sys.executable} -m pip install -e /content/ED_MRI
# !{sys.executable} -m pip install -e /content/tadred

## 1. Your starting point: the b-values of your sequence

We'll start with the b-values you designed in the previous session. These will form our **superdesign**: the full set of candidate measurements from which TADRED will select an optimised subset.

**Tasks:**

1. Load the b-values from your acquisition sequence.
2. Plot their distribution. Are they evenly spaced or concentrated in particular ranges?
3. Consider which b-values might be most informative for estimating diffusion parameters.

We'll use these b-values throughout the exercise to simulate signals, optimise the protocol, and evaluate its performance.


In [ ]:
import subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# The b-values (s/mm2) your sequence produces, one per acquisition, from the sequence session.
# TODO: point this at your own file (or paste your list). One b-value can appear several times.
BVALS_FILE = None
b_all = np.loadtxt(BVALS_FILE) if BVALS_FILE else None

if b_all is None:                       # stand-in so that the notebook runs before you have yours
    print("No b-values file given: using an example protocol. Replace it with your own!")
    b_all = np.array([0, 0, 50, 100, 150, 200, 300, 400, 500, 600, 700, 800, 1000, 1200, 1500, 2000], float)
print(len(b_all), "b-values:", b_all)


## 2. Simulate signals for your protocol

We need to generate synthetic voxels with **known parameters**, using **your b-values** and realistic noise.

We'll start with a simple **ADC model**, which has one parameter: the apparent diffusion coefficient, $D$:

$$
S(b) = \exp(-bD)
$$

Later, you can extend this to more complex models, such as IVIM, which includes an additional perfusion-related compartment.

**Tasks:**

1. Write a function `adc_signal(b, D)` to simulate noise-free diffusion signals for given b-values and diffusivities.
2. Write a function to add noise to the simulated signals.
3. Generate a large number of synthetic voxels by sampling $D$ from a plausible range.
4. Use `adc_signal` to generate a library of noisy synthetic signals and their corresponding ground-truth $D$ values.

**Noise model:** Noise and model fitting will be covered in more detail in the Day 3 talks, so a simple Rician noise model is sufficient for now:

$$
S_{\mathrm{noisy}} = \sqrt{(S+n_1)^2+n_2^2}
$$

where $n_1,n_2 \sim \mathcal{N}(0,1/\mathrm{SNR}^2)$, assuming the noise-free signal is normalised to 1 at $b=0$.


In [ ]:
def adc_signal(b, D):
    """Signal at b-values `b` (shape [nb]) for diffusivities `D` (shape [n]). Returns [n, nb]."""
    # TODO
    raise NotImplementedError

def add_noise(signal, snr):
    """Rician noise at the given SNR (S0 = 1)."""
    # TODO
    raise NotImplementedError

n_voxels, snr = 1000, 30
D_range = (0.0002, 0.003)   # mm2/s: roughly tissue-like, change it if you want to
# TODO:
# 1. draw `D_true` (shape [n_voxels]) uniformly in D_range
# 2. pass `D_true` and `b_all` to adc_signal to create library of simulated signals



## 3. Baseline: how well do **all** your b-values perform?

Before optimising the protocol, we need to establish how accurately we can estimate ADC using the full set of b-values.

**Tasks:**

1. Fit $D$ to the noisy signals for each voxel using non-linear least squares (`scipy.optimize.least_squares`, `curve_fit`, or your existing fitting code).
2. Compare the fitted $D$ values with the ground truth using RMSE (root mean squared error) or correlation.
3. Plot the fitted versus true $D$ values.

This provides a **baseline** against which we can compare the performance of smaller, optimised protocols.


In [ ]:
def fit_adc(b, signals):
    """Fit D in every row of `signals`. Returns [n]."""
    # TODO
    raise NotImplementedError

def rmse(true, est):
    # TODO
    raise NotImplementedError

signals_all = None   # TODO: simulate with adc_signal + add_noise at b_all
# D_fit_all = fit_adc(b_all, signals_all); print("RMSE, all b-values:", rmse(D_true, D_fit_all))

## 4. Let TADRED choose a subset of your b-values

TADRED optimises the acquisition protocol by selecting a subset of b-values and training a neural network to estimate the parameters from those measurements.

It requires three inputs, which the next cell saves to disk:

1. **Input:** Simulated signals at all candidate b-values.
2. **Target:** Ground-truth parameter values ($D$).
3. **Superdesign:** The full set of candidate b-values.

TADRED selects a fraction (`protocol_size`) of the candidate measurements and trains a neural network using the selected subset.

**Task:** Choose how many b-values to retain (`n_keep`) and run TADRED to optimise your protocol.


In [ ]:
from edmri.paths import CONFIG_PATH, RESULTS_PATH

np.savetxt("my_bvals.txt", b_all)
np.save("my_signals.npy", signals_all)
np.save("my_targets.npy", D_true[:, None])      # [n_voxels, n_parameters]

n_keep = None   # TODO: how many of your b-values do you keep?  (e.g. a quarter of them)
protocol_size = n_keep / len(b_all) if n_keep else 0.25

!PYTHONUNBUFFERED=1 edmri-optimise --force --example ADC \
    --config {CONFIG_PATH}/optimise_default.yaml \
    data.input=my_signals.npy data.target=my_targets.npy data.superdesign=my_bvals.txt \
    optimisation.protocol_size={protocol_size}

## 5. How well does the optimised protocol perform?

TADRED saves the selected b-values (`optimised_protocol.txt`) and the trained neural network. We now need to evaluate how well they perform on independent test data.

**Tasks:**

1. **Inspect the optimised protocol:** Load the selected b-values and plot them alongside the full protocol. Which b-values did TADRED select, and does the selection make sense?
2. **Evaluate parameter estimation:** Simulate fresh noisy signals at the selected b-values and estimate $D$ using two approaches:
   - Non-linear least-squares fitting.
   - The trained TADRED neural network (`edmri-apply`).
3. **Compare against reference protocols:** Generate two alternative protocols with the same number of measurements: a random subset and an evenly spaced subset of the original b-values. Fit $D$ using each protocol and compare their accuracy using RMSE.

**Question:** Does TADRED outperform the random and evenly spaced protocols? If not, what might explain the lack of improvement?


In [ ]:
out = Path(RESULTS_PATH, "optimisation/examples/ADC/tadred_results")
b_sel = np.loadtxt(out / "optimised_protocol.txt")
print("selected b-values:", b_sel)

# TODO: simulate fresh voxels at b_sel, fit with fit_adc, report rmse
# TODO: do the same for a random subset and an evenly-spaced subset of b_all of the same size

# The task network, applied to data simulated at the selected b-values:
# np.save("signals_sel.npy", ...)
# subprocess.run(["edmri-apply", "--force", "--example", "ADC", "--config", f"{CONFIG_PATH}/apply_default.yaml",
#                 "data.input=signals_sel.npy", f"task_network={out}/trained_task_network.pt"], check=True)
# D_net = np.load(Path(RESULTS_PATH, "application/examples/ADC/predictions.npy"))

## 6. Go further: explore what interests your team

Choose one or more of the following extensions to investigate how different factors affect protocol optimisation.

1. **Training data:** The default suggestion was to use 1,000 simulated voxels to train TADRED. Does increasing the number of training samples improve performance?
2. **Number of measurements:** Repeat Sections 4–5 for different values of `n_keep`. Plot the estimation error against the number of measurements for TADRED, random, and evenly spaced protocols. What is the minimum number of measurements needed for good accuracy?
3. **More complex models:** Extend the ADC model to IVIM, which has three parameters: $S(b) = f\exp(-bD^*) + (1-f)\exp(-bD)$. How does this affect the optimised protocol? Does TADRED select more low b-values?
4. **Noise:** Repeat the optimisation at different SNRs. Does the optimal protocol change as noise increases?
5. **Sequence design:** Do the selected b-values suggest improvements to your original acquisition protocol? Which measurements would you remove, add, or repeat?
6. **Optimisation settings:** Make a copy of `optimise_default.yaml` and experiment with the network architecture or number of reduction stages. How do these choices affect performance?

**Overall challenge:** Find the smallest subset of **your b-values** that achieves acceptable parameter estimation accuracy. Can you reduce the acquisition time without substantially compromising accuracy?
